# The EUV Imaging Spectrometer for Hinode — Implementation / 구현

**Paper**: J.L. Culhane et al., "The EUV Imaging Spectrometer for Hinode," *Solar Physics*, 243, 19–61, 2007.

This notebook implements the key instrument models from the paper:
이 노트북은 논문의 핵심 장비 모델을 구현합니다:

1. **Wavelength Dispersion Relation / 파장 분산 관계** — pixel → wavelength 변환 (Eq. 1)
2. **EIS Effective Area Model / EIS 유효 면적 모델** — 구성 요소별 효율 곱 (Eq. 5)
3. **Expected Count Rates / 예상 카운트율** — quiet Sun, active region, flare 조건 (Tables 11–13)
4. **Doppler Velocity & Nonthermal Velocity / Doppler 속도 & 비열적 속도** — 합성 선 프로파일 분석

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.special import voigt_profile

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Physical constants
C_LIGHT = 2.998e10       # Speed of light [cm/s]
K_B = 1.381e-16          # Boltzmann constant [erg/K]
M_PROTON = 1.673e-24     # Proton mass [g]

## Part 1: Wavelength Dispersion Relation / 파장 분산 관계

EIS의 CCD 픽셀 번호($p$)를 파장($\lambda$)으로 변환하는 분산 관계식 (Eq. 1):

The dispersion relation converting CCD pixel number ($p$) to wavelength ($\lambda$) (Eq. 1):

$$\lambda(p) = \lambda_0 + Ap + Bp^2$$

논문 Section 7.1에서 Penning 방전 램프의 표준선을 사용하여 결정된 계수:
Coefficients determined using standard lines from Penning discharge lamp in Section 7.1:

| Band | $\lambda_0$ [Å] | $A$ [Å/pixel] | $B$ [Å/pixel²] | Std. dev. [Å] |
|---|---|---|---|---|
| LW | 199.9389 | 0.022332 | $-1.329 \times 10^{-8}$ | 0.00415 |
| SW | 166.131 | 0.022317 | $-1.268 \times 10^{-8}$ | 0.00386 |

In [ ]:
# --- Dispersion relation coefficients (Table in Section 7.1) ---
DISP = {
    'SW': {'lam0': 166.131, 'A': 0.022317, 'B': -1.268e-8},
    'LW': {'lam0': 199.9389, 'A': 0.022332, 'B': -1.329e-8},
}


def eis_dispersion(pixel: np.ndarray, band: str) -> np.ndarray:
    """Convert pixel number to wavelength [Angstrom] using Eq. 1.

    Args:
        pixel: Pixel number (0-indexed column on CCD).
        band: 'SW' or 'LW'.

    Returns:
        Wavelength in Angstrom.
    """
    c = DISP[band]
    return c['lam0'] + c['A'] * pixel + c['B'] * pixel**2


# Compute wavelength arrays for both bands
pixels = np.arange(2048)
lam_sw = eis_dispersion(pixels, 'SW')
lam_lw = eis_dispersion(pixels, 'LW')

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, lam, band, color in zip(
    axes, [lam_sw, lam_lw], ['SW', 'LW'], ['steelblue', 'darkorange']
):
    ax.plot(pixels, lam, color=color, lw=2)
    ax.set_xlabel('Pixel number')
    ax.set_ylabel('Wavelength [Å]')
    ax.set_title(f'EIS {band} Band Dispersion')

    # Show the quadratic deviation from linear
    lam_linear = DISP[band]['lam0'] + DISP[band]['A'] * pixels
    deviation = (lam - lam_linear) * 1e3  # mÅ
    ax2 = ax.twinx()
    ax2.plot(pixels, deviation, color='gray', ls='--', alpha=0.6, lw=1)
    ax2.set_ylabel('Quadratic deviation from linear [mÅ]', color='gray')

    # Annotate range
    ax.annotate(
        f'{lam[0]:.1f} – {lam[-1]:.1f} Å',
        xy=(0.05, 0.92), xycoords='axes fraction', fontsize=11,
        bbox=dict(boxstyle='round', fc='white', alpha=0.8)
    )

plt.tight_layout()
plt.suptitle('EIS Wavelength Dispersion (Eq. 1)', y=1.02, fontsize=14)
plt.show()

# Print key numbers
print("=== Dispersion Summary ===")
for band in ['SW', 'LW']:
    lam = eis_dispersion(pixels, band)
    print(f"\n{band} band:")
    print(f"  Range: {lam[0]:.2f} – {lam[-1]:.2f} Å")
    print(f"  Linear scale: {DISP[band]['A']*1e3:.2f} mÅ/pixel "
          f"= {DISP[band]['A']/lam.mean()*C_LIGHT/1e5:.1f} km/s per pixel")
    print(f"  Quadratic term at pixel 2048: "
          f"{DISP[band]['B']*2048**2*1e3:.2f} mÅ (negligible)")

## Part 2: EIS Effective Area Model / EIS 유효 면적 모델

검출기 픽셀당 초당 등록 광자수 (Eq. 5):

Photons registered per detector pixel per second (Eq. 5):

$$N_\lambda = \phi_\lambda \cdot A \cdot \omega_d \cdot T_\text{ff}(\lambda) \cdot T_\text{spider} \cdot R_m(\lambda) \cdot E_g(\lambda) \cdot V_d(\lambda) \cdot T_\text{sf}(\lambda) \cdot E_\text{det}(\lambda)$$

유효 면적은 기하학적 면적에 모든 효율 인자를 곱한 것:

The effective area is the geometric area multiplied by all efficiency factors:

$$A_\text{eff}(\lambda) = A \cdot T_\text{ff}(\lambda) \cdot T_\text{spider} \cdot R_m(\lambda) \cdot E_g(\lambda) \cdot V_d(\lambda) \cdot T_\text{sf}(\lambda) \cdot E_\text{det}(\lambda)$$

각 구성 요소의 효율을 Gaussian 근사로 모델링하여 Figure 30의 유효 면적 곡선을 재현합니다.

We model each component's efficiency with Gaussian approximations to reproduce the effective area curves from Figure 30.

In [ ]:
# === EIS Effective Area Model (Eq. 5, Figures 28-30) ===

# Geometric mirror area [cm^2]
# 15 cm diameter usable, but aperture measured ~78 cm^2 (Fig. 27 center)
A_MIRROR = 78.0  # cm^2

# Spider/mesh open fraction (Section 4.1)
T_SPIDER = 0.80


def gaussian_efficiency(lam, peak, center, sigma):
    """Gaussian-shaped efficiency curve."""
    return peak * np.exp(-0.5 * ((lam - center) / sigma)**2)


def al_filter_transmission(lam):
    """Al filter transmittance model (Figure 5, Section 4.1).

    Two 1500 Å Al filters (front + slit). Average ~40-50% each.
    Combined product shown in Figure 29.
    """
    # Front filter: broad plateau ~45% with Al L-edge drop at 170 Å
    t_front = 0.45 * np.ones_like(lam)
    t_front[lam < 170] *= np.exp(-0.1 * (170 - lam[lam < 170]))

    # Slit filter: ~30% average (Figure 29)
    t_slit = 0.30 * np.ones_like(lam)
    t_slit[lam < 170] *= np.exp(-0.1 * (170 - lam[lam < 170]))

    return t_front, t_slit


def mirror_reflectivity(lam):
    """Mo/Si multilayer mirror reflectivity (Figure 28).

    D-shaped: two sectors optimized for SW and LW.
    Peak reflectivities: SW ~32%, LW ~23%.
    """
    r_sw = gaussian_efficiency(lam, 0.32, 190.0, 12.0)
    r_lw = gaussian_efficiency(lam, 0.23, 270.0, 12.0)
    return np.maximum(r_sw, r_lw)


def grating_efficiency(lam):
    """First-order grating diffraction efficiency (Figure 12).

    Laminar grating, 4200 lines/mm, groove depth 60 Å.
    Peak 1st order: SW 8.0% at 196 Å, LW 7.9% at 271 Å.
    """
    e_sw = gaussian_efficiency(lam, 0.080, 196.0, 12.0)
    e_lw = gaussian_efficiency(lam, 0.079, 271.0, 12.0)
    return np.maximum(e_sw, e_lw)


def ccd_qe(lam):
    """CCD quantum efficiency (Section 7.2).

    Back-illuminated thinned e2v CCD 42-20.
    Adopted values: SW 44±4%, LW 39±4% (250 Å), 37±4% (290 Å).
    """
    # Slight decline toward longer wavelengths
    qe = np.where(
        lam < 220,
        0.44 * np.ones_like(lam),
        0.44 - 0.025 * (lam - 220) / 70  # gradual decline
    )
    return np.clip(qe, 0.35, 0.44)


def vignetting(lam):
    """Vignetting factor for LW band (Section 7.2, Figure 27).

    Housing vignettes LW CCD for wavelengths > 272 Å.
    """
    v = np.ones_like(lam)
    mask = lam > 272
    v[mask] = 1.0 - 0.3 * ((lam[mask] - 272) / 18)**2
    return np.clip(v, 0.5, 1.0)


def eis_effective_area(lam):
    """Compute EIS effective area [cm^2] as a function of wavelength.

    Combines all component efficiencies following Eq. 5.
    """
    t_ff, t_sf = al_filter_transmission(lam)
    rm = mirror_reflectivity(lam)
    eg = grating_efficiency(lam)
    qe = ccd_qe(lam)
    vd = vignetting(lam)

    a_eff = A_MIRROR * T_SPIDER * t_ff * t_sf * rm * eg * vd * qe
    return a_eff


# --- Compute and plot ---
lam_full = np.linspace(165, 295, 1000)
a_eff = eis_effective_area(lam_full)

# Separate into SW and LW ranges
sw_mask = (lam_full >= 170) & (lam_full <= 210)
lw_mask = (lam_full >= 250) & (lam_full <= 290)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(lam_full[sw_mask], a_eff[sw_mask], 'b-', lw=2)
axes[0].set_xlabel('Wavelength [Å]')
axes[0].set_ylabel('Effective Area [cm²]')
axes[0].set_title('SW Effective Area (cf. Figure 30 top)')
axes[0].set_xlim(170, 210)
peak_sw = a_eff[sw_mask].max()
axes[0].annotate(f'Peak: {peak_sw:.3f} cm²', xy=(0.6, 0.9),
                 xycoords='axes fraction', fontsize=11,
                 bbox=dict(boxstyle='round', fc='lightyellow'))

axes[1].plot(lam_full[lw_mask], a_eff[lw_mask], 'r-', lw=2)
axes[1].set_xlabel('Wavelength [Å]')
axes[1].set_ylabel('Effective Area [cm²]')
axes[1].set_title('LW Effective Area (cf. Figure 30 bottom)')
axes[1].set_xlim(250, 290)
peak_lw = a_eff[lw_mask].max()
axes[1].annotate(f'Peak: {peak_lw:.3f} cm²', xy=(0.6, 0.9),
                 xycoords='axes fraction', fontsize=11,
                 bbox=dict(boxstyle='round', fc='lightyellow'))

plt.suptitle('EIS Effective Area Model (Eq. 5)', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

# Compare with paper values
print("=== Effective Area Comparison ===")
print(f"SW peak: {peak_sw:.3f} cm² (paper: 0.30 cm²)")
print(f"LW peak: {peak_lw:.3f} cm² (paper: 0.11 cm²)")

### Component Efficiency Breakdown / 구성 요소별 효율 분해

각 광학 구성 요소의 효율을 개별적으로 시각화합니다. 어느 구성 요소가 전체 처리량을 제한하는지 확인합니다.

Visualize each optical component's efficiency individually. Identify which component limits overall throughput.

In [ ]:
# === Component Efficiency Breakdown ===
lam_sw_range = np.linspace(170, 210, 300)
lam_lw_range = np.linspace(250, 290, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, lam_range, band_label in zip(
    axes, [lam_sw_range, lam_lw_range], ['SW Band', 'LW Band']
):
    t_ff, t_sf = al_filter_transmission(lam_range)
    rm = mirror_reflectivity(lam_range)
    eg = grating_efficiency(lam_range)
    qe = ccd_qe(lam_range)
    vd = vignetting(lam_range)

    ax.plot(lam_range, t_ff, 'b-', label=f'Front filter (peak {t_ff.max():.0%})', lw=1.5)
    ax.plot(lam_range, t_sf, 'b--', label=f'Slit filter (peak {t_sf.max():.0%})', lw=1.5)
    ax.plot(lam_range, rm, 'r-', label=f'Mirror refl. (peak {rm.max():.0%})', lw=1.5)
    ax.plot(lam_range, eg, 'g-', label=f'Grating eff. (peak {eg.max():.1%})', lw=1.5)
    ax.plot(lam_range, qe, 'm-', label=f'CCD QE (peak {qe.max():.0%})', lw=1.5)
    if band_label == 'LW Band':
        ax.plot(lam_range, vd, 'k--', label=f'Vignetting', lw=1.5)

    # Combined product (normalized by geometric area)
    combined = t_ff * t_sf * rm * eg * qe * vd * T_SPIDER
    ax.fill_between(lam_range, combined, alpha=0.15, color='orange')
    ax.plot(lam_range, combined, 'orange', lw=2,
            label=f'Combined (peak {combined.max():.2e})')

    ax.set_xlabel('Wavelength [Å]')
    ax.set_ylabel('Fractional efficiency')
    ax.set_title(f'{band_label} Component Efficiencies')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylim(0, 0.55)

plt.tight_layout()
plt.show()

# Identify the bottleneck
print("=== Throughput Bottleneck Analysis ===")
print("The grating 1st-order efficiency (~8%) is the dominant bottleneck.")
print("Mirror reflectivity (~23-32%) and filter transmission (~14% combined)")
print("also contribute significantly to the overall throughput loss.")

## Part 3: Expected Count Rates / 예상 카운트율

Tables 11–13에서 quiet Sun, active region, flare 조건별 주요 방출선의 예상 카운트율을 비교합니다.
논문의 카운트율은 CHIANTI V4 합성 스펙트럼과 DEM 모델에서 산출되었습니다.

Comparing expected count rates for key emission lines under quiet Sun, active region, and flare conditions from Tables 11–13. Count rates in the paper were derived from CHIANTI V4 synthetic spectra and DEM models.

In [ ]:
# === Expected Count Rates from Tables 11, 12, 13 ===
# Format: (Ion, wavelength [Å], log T [K], DN_quiet, DN_active, DN_flare)

EIS_LINES = [
    # SW band lines
    ('Fe X',     184.54, 6.00,   2.28,    25.71,       None),
    ('Fe XII',   186.88, 6.10,   4.21,    76.49,       None),
    ('Fe XI',    188.23, 6.10,   6.91,   101.04,       None),
    ('Fe XII',   193.52, 6.10,  20.60,   388.63,   14421.38),
    ('Fe XII',   195.12, 6.10,  36.63,   690.66,   25621.28),
    ('Ca XIV',   193.87, 6.50,   None,    17.82,    6307.85),
    ('Fe XXIV',  192.03, 7.20,   None,     None,  119458.12),
    ('Ca XVII',  192.82, 6.70,   None,     None,  148003.11),
    # LW band lines
    ('He II',    256.32, 4.90,   1.98,     6.24,   11568.22),
    ('Si X',     258.37, 6.10,   1.66,    30.56,       None),
    ('Fe XIV',   264.79, 6.30,   3.05,    95.20,    5591.22),
    ('Fe XV',    284.16, 6.30,   4.78,   226.17,   35546.88),
    ('Fe XVI',   262.98, 6.40,   None,    24.78,   18611.88),
]

# --- Visualization: count rates across solar conditions ---
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

conditions = [
    ('Quiet Sun', 3, 'steelblue'),
    ('Active Region', 4, 'darkorange'),
    ('Flare', 5, 'crimson'),
]

for ax, (label, col_idx, color) in zip(axes, conditions):
    ions = []
    rates = []
    wavelengths = []

    for line in EIS_LINES:
        dn = line[col_idx]
        if dn is not None:
            ions.append(f"{line[0]}\n{line[1]:.1f} Å")
            rates.append(dn)
            wavelengths.append(line[1])

    y_pos = np.arange(len(ions))
    bars = ax.barh(y_pos, rates, color=color, alpha=0.7, edgecolor='k', lw=0.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(ions, fontsize=9)
    ax.set_xlabel('Count rate [DN s⁻¹ pixel⁻¹]')
    ax.set_title(f'{label} (Table {col_idx - 2 + 11})')
    ax.set_xscale('log')

    # Annotate values
    for bar, rate in zip(bars, rates):
        ax.text(bar.get_width() * 1.1, bar.get_y() + bar.get_height()/2,
                f'{rate:.1f}', va='center', fontsize=8)

plt.suptitle('EIS Expected Count Rates (Tables 11–13)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# --- Exposure time estimation ---
print("=== Exposure Time Estimates (1-arcsec slit, S/N=10) ===")
print(f"{'Ion':<12} {'λ [Å]':>8} {'Quiet [s]':>12} {'AR [s]':>10} {'Flare [s]':>12}")
print("-" * 56)
for line in EIS_LINES:
    ion, lam, logT = line[0], line[1], line[2]
    for condition, col_idx, name in [('Quiet', 3, 'quiet'),
                                      ('AR', 4, 'ar'),
                                      ('Flare', 5, 'flare')]:
        dn = line[col_idx]
        if dn is not None and dn > 0:
            # S/N ~ sqrt(DN * t * gain) / gain for photon noise dominated
            # For S/N=10: t = (S/N)^2 / DN  (simplified)
            t_snr10 = 100.0 / dn
            if condition == 'Quiet':
                t_q = f"{t_snr10:.1f}"
            elif condition == 'AR':
                t_ar = f"{t_snr10:.2f}"
            else:
                t_fl = f"{t_snr10:.4f}"
        else:
            if condition == 'Quiet':
                t_q = "—"
            elif condition == 'AR':
                t_ar = "—"
            else:
                t_fl = "—"
    print(f"{ion:<12} {lam:>8.2f} {t_q:>12} {t_ar:>10} {t_fl:>12}")

## Part 4: Doppler Velocity & Nonthermal Velocity / Doppler 속도 & 비열적 속도

EIS의 핵심 과학 능력 중 하나는 방출선의 Doppler shift와 선폭을 분석하여 플라즈마 속도와 비열적 운동을 측정하는 것입니다.

One of EIS's key scientific capabilities is measuring plasma velocities and nonthermal motions by analyzing Doppler shifts and line widths of emission lines.

**Doppler 속도 / Doppler velocity:**
$$v = c \cdot \frac{\Delta\lambda}{\lambda_0}$$

**관측 선폭 / Observed line width:**
$$\Delta\lambda_\text{obs}^2 = \Delta\lambda_\text{inst}^2 + 4\ln 2 \cdot \frac{\lambda_0^2}{c^2}\left(\frac{2k_BT}{m_i} + \xi^2\right)$$

여기서 $\xi$는 비열적 속도입니다. 합성 EIS 선 프로파일을 생성하고 Gaussian fitting으로 속도를 복원합니다.

Where $\xi$ is the nonthermal velocity. We generate synthetic EIS line profiles and recover velocities via Gaussian fitting.

In [ ]:
# === Synthetic EIS Line Profile Generation and Analysis ===

# EIS instrumental parameters
EIS_PIXEL_SCALE = 0.0223  # Å/pixel (spectral)
EIS_INST_FWHM = 0.056     # Å (measured from He II 256 Å, Section 7.1)
EIS_INST_SIGMA = EIS_INST_FWHM / (2 * np.sqrt(2 * np.log(2)))  # Convert to sigma

# Fe XII 195.12 Å line parameters
LAM0 = 195.12     # Rest wavelength [Å]
A_FE = 56         # Iron atomic mass number
M_ION = A_FE * M_PROTON  # Ion mass [g]


def thermal_width(lam0: float, T: float, m_ion: float) -> float:
    """Compute thermal Doppler width (sigma) in Angstrom.

    Args:
        lam0: Rest wavelength [Å].
        T: Ion temperature [K].
        m_ion: Ion mass [g].

    Returns:
        Thermal sigma in Å.
    """
    v_th = np.sqrt(2 * K_B * T / m_ion)  # Thermal velocity [cm/s]
    return lam0 * v_th / C_LIGHT  # Convert to Å via Δλ/λ = v/c


def total_line_sigma(lam0: float, T: float, m_ion: float,
                     xi: float, sigma_inst: float) -> float:
    """Compute total observed line width (sigma) in Angstrom.

    Combines instrumental, thermal, and nonthermal broadening in quadrature.

    Args:
        lam0: Rest wavelength [Å].
        T: Ion temperature [K].
        m_ion: Ion mass [g].
        xi: Nonthermal velocity [cm/s].
        sigma_inst: Instrumental sigma [Å].

    Returns:
        Total sigma in Å.
    """
    sigma_thermal = thermal_width(lam0, T, m_ion)
    sigma_nonth = lam0 * xi / C_LIGHT
    sigma_phys = np.sqrt(sigma_thermal**2 + sigma_nonth**2)
    return np.sqrt(sigma_inst**2 + sigma_phys**2)


def generate_eis_profile(lam0: float, T: float, m_ion: float,
                         v_doppler: float, xi: float,
                         peak_dn: float, t_exp: float,
                         n_pixels: int = 40) -> tuple:
    """Generate a synthetic EIS emission line profile with Poisson noise.

    Args:
        lam0: Rest wavelength [Å].
        T: Ion temperature [K].
        m_ion: Ion mass [g].
        v_doppler: Line-of-sight velocity [km/s] (positive = redshift).
        xi: Nonthermal velocity [km/s].
        peak_dn: Peak count rate [DN/s/pixel].
        t_exp: Exposure time [s].
        n_pixels: Number of spectral pixels in window.

    Returns:
        Tuple of (wavelength array, noisy DN counts, true profile).
    """
    # Wavelength array centered on rest wavelength
    pix = np.arange(n_pixels) - n_pixels // 2
    lam = lam0 + pix * EIS_PIXEL_SCALE

    # Doppler-shifted center
    lam_center = lam0 * (1 + v_doppler * 1e5 / C_LIGHT)

    # Total sigma
    sigma_total = total_line_sigma(
        lam0, T, m_ion, xi * 1e5, EIS_INST_SIGMA
    )

    # Gaussian profile
    profile = peak_dn * np.exp(-0.5 * ((lam - lam_center) / sigma_total)**2)

    # Observed counts with Poisson noise
    expected_counts = profile * t_exp
    observed_counts = np.random.poisson(
        np.maximum(expected_counts, 0).astype(int)
    ).astype(float)

    return lam, observed_counts, expected_counts


def gaussian_model(lam, amplitude, center, sigma, background):
    """Gaussian model for fitting."""
    return amplitude * np.exp(-0.5 * ((lam - center) / sigma)**2) + background


def fit_eis_line(lam: np.ndarray, counts: np.ndarray,
                 lam0_guess: float) -> dict:
    """Fit a Gaussian to an EIS line profile.

    Args:
        lam: Wavelength array [Å].
        counts: Observed counts.
        lam0_guess: Initial guess for line center [Å].

    Returns:
        Dictionary with fit results.
    """
    p0 = [counts.max(), lam0_guess, 0.03, counts.min()]
    bounds = ([0, lam.min(), 0.005, -np.inf],
              [np.inf, lam.max(), 0.2, np.inf])

    popt, pcov = curve_fit(gaussian_model, lam, counts, p0=p0, bounds=bounds)
    perr = np.sqrt(np.diag(pcov))

    # Derived quantities
    v_doppler = (popt[1] - lam0_guess) / lam0_guess * C_LIGHT / 1e5  # km/s
    v_err = perr[1] / lam0_guess * C_LIGHT / 1e5

    fwhm_obs = popt[2] * 2 * np.sqrt(2 * np.log(2))

    return {
        'amplitude': popt[0], 'center': popt[1], 'sigma': popt[2],
        'background': popt[3],
        'v_doppler': v_doppler, 'v_err': v_err,
        'fwhm': fwhm_obs,
        'popt': popt, 'perr': perr,
    }


# === Demo: three scenarios ===
np.random.seed(42)

scenarios = [
    {'name': 'Quiet Sun upflow',
     'T': 1.3e6, 'v': -5.0, 'xi': 20.0,
     'peak_dn': 36.63, 't_exp': 30.0},
    {'name': 'Active Region redshift',
     'T': 1.3e6, 'v': 15.0, 'xi': 30.0,
     'peak_dn': 690.66, 't_exp': 10.0},
    {'name': 'Flare blueshift',
     'T': 2.0e6, 'v': -50.0, 'xi': 80.0,
     'peak_dn': 25621.28, 't_exp': 2.0},
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, scenario in zip(axes, scenarios):
    lam, obs, true = generate_eis_profile(
        LAM0, scenario['T'], M_ION,
        scenario['v'], scenario['xi'],
        scenario['peak_dn'], scenario['t_exp'],
    )

    # Fit
    result = fit_eis_line(lam, obs, LAM0)
    lam_fine = np.linspace(lam.min(), lam.max(), 200)
    fit_curve = gaussian_model(lam_fine, *result['popt'])

    # Plot
    ax.step(lam, obs, 'k-', where='mid', lw=1, label='Observed (Poisson)')
    ax.plot(lam_fine, fit_curve, 'r-', lw=2, label='Gaussian fit')
    ax.axvline(LAM0, color='gray', ls=':', alpha=0.5, label=f'Rest λ={LAM0} Å')
    ax.axvline(result['center'], color='r', ls='--', alpha=0.5)

    ax.set_xlabel('Wavelength [Å]')
    ax.set_ylabel('Counts [DN]')
    ax.set_title(scenario['name'])
    ax.legend(fontsize=8)

    # Annotation
    info = (f"Input: v={scenario['v']:.0f} km/s, ξ={scenario['xi']:.0f} km/s\n"
            f"Fit:   v={result['v_doppler']:.1f}±{result['v_err']:.1f} km/s\n"
            f"FWHM={result['fwhm']*1e3:.1f} mÅ")
    ax.annotate(info, xy=(0.03, 0.97), xycoords='axes fraction',
                fontsize=9, va='top', family='monospace',
                bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

plt.suptitle('Synthetic EIS Fe XII 195.12 Å Line Profiles', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Nonthermal Velocity Extraction / 비열적 속도 추출

관측된 선폭에서 장비 프로파일과 열적 기여를 제거하여 비열적 속도 $\xi$를 추출합니다:

Extract nonthermal velocity $\xi$ by removing instrumental and thermal contributions from the observed line width:

$$\xi = \frac{c}{\lambda_0}\sqrt{\sigma_\text{obs}^2 - \sigma_\text{inst}^2 - \sigma_\text{thermal}^2}$$

In [ ]:
# === Nonthermal Velocity Extraction Demo ===

def extract_nonthermal_velocity(sigma_obs: float, lam0: float,
                                T: float, m_ion: float) -> float:
    """Extract nonthermal velocity from observed line width.

    Args:
        sigma_obs: Observed Gaussian sigma [Å].
        lam0: Rest wavelength [Å].
        T: Assumed ion temperature [K].
        m_ion: Ion mass [g].

    Returns:
        Nonthermal velocity [km/s].
    """
    sigma_thermal = thermal_width(lam0, T, m_ion)
    sigma_excess_sq = sigma_obs**2 - EIS_INST_SIGMA**2 - sigma_thermal**2

    if sigma_excess_sq > 0:
        xi_cms = C_LIGHT / lam0 * np.sqrt(sigma_excess_sq)
        return xi_cms / 1e5  # km/s
    else:
        return 0.0


# Sweep over a range of input nonthermal velocities and verify recovery
np.random.seed(123)
xi_input = np.arange(0, 101, 5)  # km/s
xi_recovered = []
xi_errors = []

T_test = 1.3e6  # Fe XII formation temperature
n_trials = 50

for xi in xi_input:
    recovered_trials = []
    for _ in range(n_trials):
        lam, obs, _ = generate_eis_profile(
            LAM0, T_test, M_ION,
            v_doppler=0.0, xi=float(xi),
            peak_dn=690.66, t_exp=10.0,
        )
        try:
            result = fit_eis_line(lam, obs, LAM0)
            xi_rec = extract_nonthermal_velocity(
                result['sigma'], LAM0, T_test, M_ION
            )
            recovered_trials.append(xi_rec)
        except RuntimeError:
            pass

    xi_recovered.append(np.mean(recovered_trials))
    xi_errors.append(np.std(recovered_trials))

xi_recovered = np.array(xi_recovered)
xi_errors = np.array(xi_errors)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: input vs recovered
ax = axes[0]
ax.errorbar(xi_input, xi_recovered, yerr=xi_errors, fmt='o-', color='steelblue',
            capsize=3, markersize=5, label='Recovered')
ax.plot([0, 100], [0, 100], 'k--', alpha=0.5, label='1:1 line')
ax.set_xlabel('Input ξ [km/s]')
ax.set_ylabel('Recovered ξ [km/s]')
ax.set_title('Nonthermal Velocity Recovery Test')
ax.legend()
ax.set_aspect('equal')
ax.set_xlim(-5, 105)
ax.set_ylim(-5, 105)

# Right: line width budget
ax = axes[1]
xi_range = np.linspace(0, 100, 200)  # km/s
sigma_inst_arr = np.full_like(xi_range, EIS_INST_SIGMA)
sigma_th_arr = np.full_like(xi_range, thermal_width(LAM0, T_test, M_ION))
sigma_nonth_arr = LAM0 * xi_range * 1e5 / C_LIGHT
sigma_total_arr = np.sqrt(sigma_inst_arr**2 + sigma_th_arr**2 + sigma_nonth_arr**2)

# Convert to FWHM in mÅ
fwhm_factor = 2 * np.sqrt(2 * np.log(2))
ax.plot(xi_range, sigma_inst_arr * fwhm_factor * 1e3, 'b--',
        label=f'Instrumental ({EIS_INST_FWHM*1e3:.0f} mÅ)', lw=1.5)
ax.plot(xi_range, sigma_th_arr * fwhm_factor * 1e3, 'r--',
        label=f'Thermal @ {T_test/1e6:.1f} MK', lw=1.5)
ax.plot(xi_range, sigma_nonth_arr * fwhm_factor * 1e3, 'g--',
        label='Nonthermal', lw=1.5)
ax.plot(xi_range, sigma_total_arr * fwhm_factor * 1e3, 'k-',
        label='Total observed', lw=2)

ax.set_xlabel('Nonthermal velocity ξ [km/s]')
ax.set_ylabel('FWHM [mÅ]')
ax.set_title('Fe XII 195.12 Å Line Width Budget')
ax.legend(fontsize=9)
ax.axhline(EIS_INST_FWHM * 1e3, color='gray', ls=':', alpha=0.3)

plt.tight_layout()
plt.show()

print("=== Key Line Width Numbers ===")
print(f"Instrumental FWHM: {EIS_INST_FWHM*1e3:.1f} mÅ "
      f"({EIS_INST_FWHM/LAM0*C_LIGHT/1e5:.1f} km/s)")
print(f"Thermal FWHM (Fe XII, 1.3 MK): "
      f"{thermal_width(LAM0, T_test, M_ION)*fwhm_factor*1e3:.1f} mÅ "
      f"({np.sqrt(2*K_B*T_test/M_ION)/1e5:.1f} km/s)")
print(f"EIS can measure ξ ≳ 15 km/s reliably (where nonthermal > instrumental)")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| **Wavelength dispersion** / 파장 분산 | 2차 다항식 $\lambda(p) = \lambda_0 + Ap + Bp^2$, 사실상 선형 (~22 mÅ/pixel) | 동일한 방법론, 궤도 열 변형 보정 추가 (EIS 소프트웨어 `eis_prep`) |
| **Effective area** / 유효 면적 | Normal incidence multilayer: peak 0.30 cm² (SW), 0.11 cm² (LW) | Solar Orbiter/SPICE: 유사 설계이나 더 넓은 파장 범위 (70.4–79.0 nm, 97.3–104.9 nm) |
| **Count rate prediction** / 카운트율 예측 | CHIANTI V4 + DEM 모델로 합성 스펙트럼 생성 | CHIANTI V11+ 사용, 원자 데이터 크게 개선, Python eispac 패키지 |
| **Doppler velocity** / Doppler 속도 | Gaussian fitting으로 ±5 km/s 정밀도 | 동일하나 orbital variation 보정이 핵심 (궤도 주기 열 변형) |
| **Nonthermal velocity** / 비열적 속도 | 선폭에서 장비+열적 기여 제거, ±25 km/s | 동일 방법론, EIS 장비 프로파일의 비대칭성 보정 추가 |
| **CCD detector** / CCD 검출기 | Back-illuminated e2v CCD 42-20 (QE ~40%) | APS/CMOS 센서로 전환 추세 (차세대 미션: MUSE 등) |